# Building a Multi-Agent AI Creative Studio
## with Google ADK and the Agent-to-Agent (A2A) Protocol

**Workshop | GCP Cloud Shell**

---

In this workshop you will build a **distributed multi-agent orchestration system** that creates complete social media campaigns.

You will:
- Write **5 specialist AI agents** (Brand Strategist, Copywriter, Designer, Critic, Project Manager)
- Build an **orchestrator** (Creative Director) that coordinates them
- Expose each specialist as an **A2A (Agent-to-Agent) service** over HTTPS
- Deploy specialists to **Google Cloud Run** and the orchestrator to **Vertex AI Agent Engine**
- Run an end-to-end campaign creation workflow

**Duration:** ~2.5 hours  
**Difficulty:** Intermediate

---

## Architecture

```
                        ┌─────────────────────────────────────────────────┐
                        │          Vertex AI Agent Engine                 │
                        │                                                 │
  User Request ────────►│         Creative Director                       │
                        │    (Orchestrator / LLM Agent)                  │
                        │                                                 │
                        └──────┬──────┬──────┬──────┬────────────────────┘
                               │ A2A  │ A2A  │ A2A  │ A2A   │ A2A
                               ▼      ▼      ▼      ▼       ▼
                        ┌──────┐ ┌────┐ ┌────┐ ┌──────┐ ┌────────┐
                        │Brand │ │Copy│ │Des-│ │Critic│ │Project │
                        │Strat.│ │writ│ │ign-│ │      │ │Manager │
                        │      │ │er  │ │er  │ │      │ │(+Notion│
                        └──────┘ └────┘ └────┘ └──────┘ │  MCP)  │
                         Cloud    Cloud  Cloud  Cloud    └────────┘
                          Run      Run    Run    Run       Cloud Run
```

**Communication:** Each specialist exposes an A2A endpoint. The Creative Director discovers them via `/.well-known/agent.json` agent cards and delegates tasks using the A2A JSON-RPC protocol.

---

## What You'll Learn

| Concept | Description |
|---|---|
| **Google ADK** | Build LLM agents with `Agent`, tools, and instructions |
| **A2A Protocol** | Expose agents as standardized HTTPS services |
| **RemoteA2aAgent** | Call remote agents as local tools via `AgentTool` |
| **Cloud Run** | Deploy containerized agents at scale |
| **Agent Engine** | Host the orchestrator with session management |
| **MCP Integration** | Connect to external services (Notion) via Model Context Protocol |
| **Context Compaction** | Handle long multi-agent workflows within token limits |

## Prerequisites

Before starting, make sure you have:

- [ ] A **Google Cloud project** with billing enabled
- [ ] **Owner** or **Editor** IAM role on the project
- [ ] This notebook open in **GCP Cloud Shell** (recommended) or Google Colab

> **Cloud Shell:** Go to [console.cloud.google.com](https://console.cloud.google.com) → Click the **Activate Cloud Shell** button (`>_`) in the top toolbar.
> Then open the **Cloud Shell Editor** and upload or clone this notebook.

---
## Step 1: Authenticate and Configure GCP

In Cloud Shell, authentication is automatic. Run the cells below to configure your project.

In [ ]:
# In Cloud Shell, run this to confirm you are authenticated
!gcloud auth list

In [ ]:
# Set your GCP Project ID
# Replace with your actual project ID
PROJECT_ID = "your-project-id"  # <-- CHANGE THIS
REGION = "us-central1"

import os
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION

!gcloud config set project {PROJECT_ID}
print(f"Project set to: {PROJECT_ID}")
print(f"Region: {REGION}")

In [ ]:
# Set up Application Default Credentials (required for Vertex AI)
# In Cloud Shell this is already done; in Colab run the line below
# !gcloud auth application-default login

# Verify credentials
!gcloud auth application-default print-access-token > /dev/null && echo "Credentials OK"

---
## Step 2: Enable Required GCP APIs

The system needs several GCP services. Enable them all at once:

In [ ]:
%%bash
echo "Enabling GCP APIs (this may take 2-3 minutes)..."

gcloud services enable \
    aiplatform.googleapis.com \
    run.googleapis.com \
    cloudbuild.googleapis.com \
    artifactregistry.googleapis.com \
    generativelanguage.googleapis.com \
    iam.googleapis.com \
    cloudresourcemanager.googleapis.com

echo "All APIs enabled!"

---
## Step 3: Clone the Repository

Clone the AI Creative Studio repository to your Cloud Shell environment:

In [ ]:
%%bash
# Clone the repository
# Replace with your actual repo URL if you forked it
REPO_URL="https://github.com/YOUR_USERNAME/ai-creative-studio.git"  # <-- CHANGE THIS

git clone $REPO_URL ~/ai-creative-studio
echo "Repository cloned to ~/ai-creative-studio"

In [ ]:
# Change to the project directory
import os
os.chdir(os.path.expanduser("~/ai-creative-studio"))
!pwd
!ls -la

In [ ]:
# Explore the project structure
!find . -type f -name '*.py' | grep -v __pycache__ | sort

**Project structure explained:**

```
ai-creative-studio/
├── agents/
│   ├── brand_strategist/   # Market research with Google Search
│   ├── copywriter/         # Instagram caption creation
│   ├── designer/           # Visual concept & Imagen prompts
│   ├── critic/             # Quality review with scoring
│   ├── project_manager/    # Timeline + Notion MCP integration
│   └── creative_director/  # Orchestrator (calls all specialists)
├── deploy/                 # Deployment scripts
├── tools/                  # A2A inspector tool
├── .env.example            # Environment variable template
└── workshop/               # This notebook!
```

---
## Step 4: Install Dependencies

Install the Python packages required for the project:

In [ ]:
# Install core dependencies
!pip install -q \
    "google-adk[a2a]==1.20.0" \
    "google-genai>=1.51.0" \
    "uvicorn[standard]>=0.25.0" \
    "python-dotenv>=1.0.0"

print("Dependencies installed!")

In [ ]:
# Verify ADK installation
import google.adk
print(f"Google ADK version: {google.adk.__version__}")

---
## Step 5: Configure Environment Variables

The project uses environment variables for all configuration. Let's set them up:

In [ ]:
# Get a Gemini API key from https://aistudio.google.com/app/apikey
# Paste it below
GOOGLE_API_KEY = "your-gemini-api-key"  # <-- CHANGE THIS

# Write the .env file
env_content = f"""# GCP Configuration
GCP_PROJECT_ID={PROJECT_ID}
GCP_REGION={REGION}
PROJECT_ID={PROJECT_ID}
LOCATION={REGION}

# Gemini API
GOOGLE_API_KEY={GOOGLE_API_KEY}

# Agent URLs (will be filled after deployment)
COPYWRITER_AGENT_URL=
CRITIC_AGENT_URL=
DESIGNER_AGENT_URL=
PM_AGENT_URL=
STRATEGIST_AGENT_URL=

# Local Development
HOST=0.0.0.0
PORT=8080
PROTOCOL=http
PUBLIC_HOST=localhost
PUBLIC_PORT=8080
"""

with open(".env", "w") as f:
    f.write(env_content)

print(".env file created!")
print("\nContents (sensitive values masked):")
!grep -v 'API_KEY\|TOKEN' .env

In [ ]:
# Load environment variables into the current session
from dotenv import load_dotenv
load_dotenv(override=True)

import os
print("Environment loaded:")
print(f"  PROJECT_ID: {os.getenv('PROJECT_ID')}")
print(f"  REGION: {os.getenv('LOCATION')}")
print(f"  API_KEY set: {'Yes' if os.getenv('GOOGLE_API_KEY') else 'No'}")

---
## Step 6: The A2A Protocol — Key Concepts

Before writing agents, let's understand the **Agent-to-Agent (A2A) protocol**:

### How A2A Works

1. **Agent Card** — Each agent publishes a JSON descriptor at `/.well-known/agent.json`:
   ```json
   {
     "name": "brand_strategist",
     "description": "Market research and competitive analysis",
     "url": "https://brand-strategist-xyz.run.app",
     "capabilities": {"streaming": true},
     "skills": [{"id": "market_research", "description": "..."}]
   }
   ```

2. **Discovery** — The orchestrator fetches agent cards to discover capabilities.

3. **Invocation** — Agents communicate via JSON-RPC messages over HTTPS:
   ```
   POST /a2a/{agent_name}
   {"jsonrpc": "2.0", "method": "tasks/send", "params": {"message": {...}}}
   ```

4. **Expose with ADK** — `to_a2a()` wraps any ADK agent in an A2A-compliant FastAPI app:
   ```python
   from google.adk.a2a.utils.agent_to_a2a import to_a2a
   a2a_app = to_a2a(root_agent, host="my-service.run.app", port=443, protocol="https")
   ```

### Why A2A?

- **Framework agnostic** — ADK, LangGraph, CrewAI agents can all interoperate
- **Discoverable** — Agents self-describe their capabilities
- **Scalable** — Each agent is an independent HTTPS service
- **Secure** — Standard HTTPS + IAM authentication

---
## Step 7: Build the Brand Strategist Agent

The Brand Strategist is the first specialist. It uses **Google Search** to research markets, competitors, and trends.

**Key ADK concepts introduced:**
- `Agent` with an `instruction` (system prompt)
- Built-in `google_search` tool
- `to_a2a()` for A2A exposure
- Environment-driven public URL configuration

In [ ]:
# Read and display the Brand Strategist agent code
with open("agents/brand_strategist/agent.py") as f:
    print(f.read())

**Key points in the Brand Strategist:**

```python
# 1. Import the ADK Agent and Google Search tool
from google.adk.agents import Agent
from google.adk.tools import google_search

# 2. Define the agent — instruction = system prompt
root_agent = Agent(
    name="brand_strategist",
    model="gemini-2.5-flash",
    instruction=SYSTEM_INSTRUCTION,  # Role definition + output format
    tools=[google_search],           # Built-in web search
)

# 3. Expose as A2A service in __main__
a2a_app = to_a2a(root_agent, host=PUBLIC_HOST, port=PUBLIC_PORT, protocol=PROTOCOL)
uvicorn.run(a2a_app, host=HOST, port=PORT)
```

**PUBLIC_HOST vs HOST:** The agent *listens* on `HOST:PORT` (e.g., `0.0.0.0:8080`), but its agent card advertises `PUBLIC_HOST:PUBLIC_PORT` (e.g., the Cloud Run URL on port 443). This split is critical for containerized deployments.

---
## Step 8: Build the Copywriter Agent

The Copywriter creates Instagram captions. It **reads context from conversation history** — the Brand Strategist's insights are passed to it by the orchestrator.

**Key concept:** Agents don't share memory. The orchestrator explicitly includes prior outputs in each agent's prompt.

In [ ]:
with open("agents/copywriter/agent.py") as f:
    print(f.read())

**Key design decision — no tools needed:**

```python
agent = Agent(
    name="copywriter",
    model="gemini-2.5-flash",
    instruction=SYSTEM_INSTRUCTION,
    # No tools= needed — LLM handles this task directly
)
```

Unlike the Brand Strategist (which needs Google Search), the Copywriter only needs the LLM's language capabilities. Keep agents focused — don't add tools they don't need.

---
## Step 9: Build the Designer Agent

The Designer creates **Imagen-ready image generation prompts** for each caption. It reads both the Brand Strategist's insights and the Copywriter's captions from the conversation context.

In [ ]:
with open("agents/designer/agent.py") as f:
    print(f.read())

**Output format structured for downstream processing:**

```
For Caption 1: "Every sip saves the planet"
Concept A: Minimalist Nature
  Prompt: Product shot of reusable water bottle against lush forest backdrop,
          morning mist, golden hour lighting, 1080x1080...
  Colors: Earth tones (forest green, warm brown, cream)
  Mood:   Serene, aspirational
```

In a production system, these prompts would be sent directly to the Vertex AI Imagen API.

---
## Step 10: Build the Critic Agent (Quality Loop)

The Critic introduces a **quality control loop**. It reviews posts and visuals, assigns scores, and returns structured feedback with `APPROVED` or `NEEDS_REVISION` status.

The orchestrator reads these statuses and triggers revisions automatically — up to **1 revision round** per deliverable (to prevent infinite loops).

In [ ]:
with open("agents/critic/agent.py") as f:
    print(f.read())

**Structured output for machine-readable parsing:**

```
POSTS REVIEW:
- Score: 6/10
- Status: NEEDS_REVISION    <-- Orchestrator reads this
- Suggestions: Strengthen CTAs, more professional tone

VISUALS REVIEW:
- Score: 8/10
- Status: APPROVED          <-- Orchestrator skips designer revision

OVERALL ASSESSMENT:
- All Approved: NO          <-- Orchestrator triggers copywriter revision
```

**Scoring guide:**
- 9-10 → APPROVED (publish as-is)
- 7-8 → APPROVED (minor issues, acceptable)
- 5-6 → NEEDS_REVISION (has potential, improve)
- 1-4 → NEEDS_REVISION (significant issues)

---
## Step 11: Build the Project Manager Agent (with Notion MCP)

The Project Manager creates timelines and tasks. It optionally integrates with **Notion** via the **Model Context Protocol (MCP)**.

**Key ADK concept: MCP Toolset**

MCP (Model Context Protocol) lets agents connect to external tools as standardized stdio servers. The Notion MCP server exposes Notion API calls as tools the LLM can invoke directly.

In [ ]:
with open("agents/project_manager/agent.py") as f:
    content = f.read()
    # Show only the key agent creation logic (first 100 lines)
    print("\n".join(content.split("\n")[:100]))
    print("\n... [truncated for readability] ...")

**MCP integration pattern:**

```python
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters

# Connect to Notion MCP server via stdio
server_params = StdioServerParameters(
    command="notion-mcp-server",
    env={"NOTION_TOKEN": notion_api_key}
)

notion_toolset = McpToolset(
    connection_params=StdioConnectionParams(server_params=server_params)
)

# Agent gets Notion API tools automatically!
agent = Agent(
    name="project_manager",
    model="gemini-2.5-flash",
    instruction=get_system_instruction(database_id=notion_database_id),
    tools=[notion_toolset],  # MCP tools exposed to the LLM
)
```

**Graceful degradation:** If `NOTION_API_KEY` is not set, the agent creates text-based timelines without Notion. This is the recommended pattern — never hard-fail on optional integrations.

---
## Step 12: Build the Creative Director Orchestrator

The Creative Director is the **master orchestrator**. It:

1. Classifies request complexity (simple vs. full campaign)
2. Plans the execution sequence
3. Delegates to specialists via `RemoteA2aAgent` + `AgentTool`
4. Parses Critic feedback and triggers revisions
5. Compacts context to handle long workflows

**Key ADK concepts:**
- `RemoteA2aAgent` — wraps a remote A2A service as a local agent
- `AgentTool` — exposes an agent as a callable tool for the LLM
- `App` + `EventsCompactionConfig` — manages token limits in multi-step workflows

In [ ]:
with open("agents/creative_director/agent.py") as f:
    content = f.read()
    # Show key agent creation function (skip the long instruction)
    lines = content.split("\n")
    # Find create_creative_director function
    start = next(i for i, l in enumerate(lines) if "def create_creative_director" in l)
    print("\n".join(lines[start:start+120]))

**The `RemoteA2aAgent` + `AgentTool` pattern:**

```python
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.tools.agent_tool import AgentTool

# 1. Wrap a remote agent service
strategist_agent = RemoteA2aAgent(
    name="brand_strategist",
    description="Market research and trend analysis",
    agent_card=f"{strategist_url}/.well-known/agent.json",  # Discovers capabilities
)

# 2. Expose it as a tool the LLM can call
agent_tools.append(AgentTool(agent=strategist_agent))

# 3. Orchestrator uses all specialist tools
orchestrator = Agent(
    name="creative_director",
    model="gemini-2.5-flash",
    instruction=system_instruction,
    tools=agent_tools,  # LLM decides when to call each agent
)
```

**Context compaction for long workflows:**

```python
from google.adk.apps import App
from google.adk.apps.app import EventsCompactionConfig
from google.adk.apps.llm_event_summarizer import LlmEventSummarizer

# Summarize context after every 3 agents to avoid token limits
compaction_config = EventsCompactionConfig(
    summarizer=LlmEventSummarizer(llm=Gemini(model_id="gemini-2.5-flash")),
    compaction_interval=3,  # Summarize after 3 completed agents
    overlap_size=1,         # Keep last agent's output in full
)

app = App(
    name="creative_director",
    root_agent=agent,
    events_compaction_config=compaction_config,
)
```

---
## Step 13: Test Locally with ADK Web UI

Before deploying, test individual agents locally using the ADK web interface.

> **Note:** This step requires an interactive terminal. In Cloud Shell, run these commands in the **terminal** tab (not in the notebook).

**Option A — Test a single specialist:**

```bash
cd ~/ai-creative-studio/agents/brand_strategist
source ../../.env  # or: export GOOGLE_API_KEY=...

adk web
# Opens: http://localhost:8000
# Click "Web Preview" in Cloud Shell toolbar → "Preview on port 8000"
```

**Option B — Test the full orchestrator locally (requires agents deployed or running locally):**

```bash
cd ~/ai-creative-studio/agents/creative_director
source ../../.env

adk web
```

**Try these test prompts in the ADK UI:**

- Simple: `"Research the eco-friendly water bottle market"`
- Single task: `"Write 3 Instagram captions for a luxury watch brand"`
- Full campaign: `"Create a complete Instagram campaign for GreenBrew, an eco-friendly coffee brand targeting Gen-Z"`

In [ ]:
# Verify ADK CLI is available
!adk --version

In [ ]:
# Quick programmatic test of a single agent (Brand Strategist)
# This runs directly without the web UI
import asyncio
import os
import sys

sys.path.insert(0, os.path.expanduser("~/ai-creative-studio"))
os.chdir(os.path.expanduser("~/ai-creative-studio/agents/brand_strategist"))

from dotenv import load_dotenv
load_dotenv("../../.env")

from google.adk import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Import the Brand Strategist agent
from agent import root_agent

async def test_brand_strategist():
    session_service = InMemorySessionService()
    runner = Runner(app_name="workshop", agent=root_agent, session_service=session_service)

    await session_service.create_session(
        app_name="workshop", user_id="user1", session_id="session1"
    )

    brief = """Briefly research the eco-friendly reusable bottle market:
    - Target: health-conscious millennials
    - Platform: Instagram
    Keep response concise (2-3 bullet points per section)."""

    print("Brand Strategist working...\n")
    async for event in runner.run_async(
        user_id="user1",
        session_id="session1",
        new_message=types.Content(parts=[types.Part(text=brief)]),
    ):
        if hasattr(event, "content") and event.content:
            for part in event.content.parts:
                if hasattr(part, "text") and part.text:
                    print(part.text, end="", flush=True)

    await runner.close()
    print("\n\nDone!")

asyncio.run(test_brand_strategist())

---
## Step 14: Deploy Specialist Agents to Cloud Run

Each specialist is deployed as an independent Cloud Run service. The deployment script:

1. Uses `gcloud run deploy --source .` (builds Docker container automatically via Cloud Build)
2. Sets environment variables on each service
3. Returns the public HTTPS URL for each agent

**Dockerfile pattern (same for all specialists):**

```dockerfile
FROM python:3.12-slim
WORKDIR /app
RUN apt-get update && apt-get install -y --no-install-recommends gcc curl

# Fast dependency install with uv
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv
COPY requirements.txt .
RUN uv pip install --system --no-cache -r requirements.txt

COPY . .
RUN useradd -m -u 1000 appuser && chown -R appuser:appuser /app
USER appuser

ENV PYTHONUNBUFFERED=1 PORT=8080 HOST=0.0.0.0
EXPOSE 8080
CMD ["python", "agent.py"]
```

**A2A public URL configuration:**
- `HOST=0.0.0.0` / `PORT=8080` — where the container listens
- `PUBLIC_HOST=<cloud-run-url>` / `PUBLIC_PORT=443` / `PROTOCOL=https` — what the agent card advertises

In [ ]:
%%bash
# Step 14a: Deploy the Brand Strategist to Cloud Run
# This will take ~3-5 minutes (Cloud Build builds and pushes the container)

source .env

echo "Deploying Brand Strategist..."
STRATEGIST_URL=$(gcloud run deploy brand-strategist-agent \
    --source agents/brand_strategist \
    --region ${GCP_REGION:-us-central1} \
    --allow-unauthenticated \
    --set-env-vars "GOOGLE_API_KEY=${GOOGLE_API_KEY},PROTOCOL=https,PUBLIC_PORT=443" \
    --format='value(status.url)')

echo "Brand Strategist deployed at: $STRATEGIST_URL"

# Update .env with the URL
sed -i "s|STRATEGIST_AGENT_URL=.*|STRATEGIST_AGENT_URL=${STRATEGIST_URL}|" .env
echo "URL saved to .env"

In [ ]:
%%bash
# Step 14b: Deploy remaining specialists in parallel
# This uses the Python deploy script which deploys all 5 in parallel
# Total time: ~5-8 minutes

source .env

echo "Deploying all specialist agents in parallel..."
python deploy/deploy_all_specialists.py

echo ""
echo "All specialists deployed!"

In [ ]:
%%bash
# Reload .env to get the deployed URLs
set -a && source .env && set +a

echo "Deployed agent URLs:"
echo "  Brand Strategist: $STRATEGIST_AGENT_URL"
echo "  Copywriter:       $COPYWRITER_AGENT_URL"
echo "  Designer:         $DESIGNER_AGENT_URL"
echo "  Critic:           $CRITIC_AGENT_URL"
echo "  Project Manager:  $PM_AGENT_URL"

---
## Step 15: Verify Specialist Deployments

Check each agent is live by fetching its **agent card** — the A2A discovery endpoint:

In [ ]:
import json
import urllib.request
from dotenv import load_dotenv

# Reload env with the deployed URLs
load_dotenv(override=True)

agents = {
    "Brand Strategist": os.getenv("STRATEGIST_AGENT_URL"),
    "Copywriter": os.getenv("COPYWRITER_AGENT_URL"),
    "Designer": os.getenv("DESIGNER_AGENT_URL"),
    "Critic": os.getenv("CRITIC_AGENT_URL"),
    "Project Manager": os.getenv("PM_AGENT_URL"),
}

print("Checking agent cards...\n")
for name, url in agents.items():
    if not url:
        print(f"  {name}: URL not set")
        continue
    try:
        card_url = f"{url.rstrip('/')}/.well-known/agent.json"
        with urllib.request.urlopen(card_url, timeout=10) as resp:
            card = json.loads(resp.read())
        print(f"  {name}: OK")
        print(f"    Name: {card.get('name')}")
        print(f"    URL: {card.get('url')}")
        print()
    except Exception as e:
        print(f"  {name}: ERROR - {e}")

In [ ]:
%%bash
# Quick smoke test — send a simple message to the Brand Strategist
source .env

echo "Testing Brand Strategist..."
curl -s -X POST "${STRATEGIST_AGENT_URL}/a2a/brand_strategist" \
  -H "Content-Type: application/json" \
  -d '{
    "jsonrpc": "2.0",
    "method": "tasks/send",
    "id": "test-1",
    "params": {
      "message": {
        "role": "user",
        "parts": [{"text": "Brief one-sentence intro of your capabilities."}]
      }
    }
  }' | python3 -c "
import sys, json
r = json.load(sys.stdin)
result = r.get('result', {})
parts = result.get('status', {}).get('message', {}).get('parts', [])
for p in parts:
    if 'text' in p:
        print(p['text'])
"

echo ""
echo "Brand Strategist is live!"

---
## Step 16: Deploy the Creative Director to Vertex AI Agent Engine

The orchestrator is deployed to **Vertex AI Agent Engine**, which provides:
- Managed session state
- Automatic scaling
- Built-in tracing and logging
- ADK-native integration

**How Agent Engine deployment works:**

```python
import vertexai
from vertexai.preview import reasoning_engines

vertexai.init(project=PROJECT_ID, location=REGION)

# Package the App (not Agent directly — App has compaction config)
remote_app = reasoning_engines.ReasoningEngine.create(
    reasoning_engines.AdkApp(app=root_app, enable_tracing=True),
    requirements=["google-adk[a2a]==1.20.0", "google-genai>=1.51.0"],
    display_name="Creative Director",
    extra_packages=["./agents/creative_director"],
)
```

In [ ]:
%%bash
# Load deployed agent URLs into environment
source .env

# Verify all specialist URLs are set before deploying orchestrator
missing=0
for var in STRATEGIST_AGENT_URL COPYWRITER_AGENT_URL DESIGNER_AGENT_URL CRITIC_AGENT_URL PM_AGENT_URL; do
    val=$(eval echo \$$var)
    if [ -z "$val" ]; then
        echo "ERROR: $var is not set"
        missing=1
    else
        echo "OK: $var = $val"
    fi
done

if [ $missing -eq 1 ]; then
    echo ""
    echo "Please complete Step 14 before deploying the orchestrator"
else
    echo ""
    echo "All agent URLs configured. Ready to deploy Creative Director!"
fi

In [ ]:
%%bash
# Deploy the Creative Director orchestrator to Vertex AI Agent Engine
# This will take ~5-10 minutes
source .env

echo "Deploying Creative Director to Agent Engine..."
python deploy/deploy_orchestrator.py --action deploy

echo ""
echo "Creative Director deployed!"

In [ ]:
%%bash
# Reload .env to get the Agent Engine resource name
source .env

echo "Agent Engine deployment info:"
echo "  Resource: $AGENT_ENGINE_RESOURCE_NAME"
echo "  ID:       $AGENT_ENGINE_ID"

---
## Step 17: Run an End-to-End Campaign

The entire system is now deployed. Let's run a complete campaign creation workflow!

This will trigger the full 5-agent sequence:
1. Brand Strategist → market research
2. Copywriter → 5 Instagram posts
3. Designer → image concepts
4. Critic → quality review (may trigger revisions)
5. Project Manager → timeline

In [ ]:
import vertexai
from vertexai.preview import reasoning_engines
from dotenv import load_dotenv
import os

load_dotenv(override=True)

PROJECT_ID = os.getenv("PROJECT_ID")
REGION = os.getenv("LOCATION", "us-central1")
AGENT_ENGINE_ID = os.getenv("AGENT_ENGINE_ID")

if not AGENT_ENGINE_ID:
    raise ValueError("AGENT_ENGINE_ID not set. Complete Step 16 first.")

# Connect to the deployed Agent Engine
vertexai.init(project=PROJECT_ID, location=REGION)

agent_engine = reasoning_engines.ReasoningEngine(
    f"projects/{PROJECT_ID}/locations/{REGION}/reasoningEngines/{AGENT_ENGINE_ID}"
)

print(f"Connected to Agent Engine: {AGENT_ENGINE_ID}")

In [ ]:
# Create a session for this workshop run
session = agent_engine.create_session(user_id="workshop-participant")
session_id = session["id"]
print(f"Session created: {session_id}")

In [ ]:
# Run a complete campaign — this will take 3-5 minutes
campaign_brief = """
Create a complete Instagram campaign for:
- Product: EcoFlow Smart Water Bottle (tracks hydration, keeps drinks cold 24h)
- Target Audience: Health-conscious millennials, 25-35 years old
- Platform: Instagram
- Goal: Brand awareness + drive website traffic
- Brand Voice: Motivational, clean, science-backed
- Budget: $3,000
- Timeline: Launch in 2 weeks
"""

print("Sending campaign brief to Creative Director...")
print("The orchestrator will coordinate all 5 specialist agents.")
print("This takes 3-5 minutes.\n")
print("=" * 60)

for event in agent_engine.stream_query(
    user_id="workshop-participant",
    session_id=session_id,
    message=campaign_brief,
):
    if "content" in event and "parts" in event["content"]:
        for part in event["content"]["parts"]:
            if "text" in part:
                print(part["text"], end="", flush=True)

print("\n" + "=" * 60)
print("Campaign creation complete!")

In [ ]:
# Try a simpler, single-agent request
print("Testing single-agent routing...\n")
print("Prompt: 'Research the luxury skincare market on Instagram'")
print("Expected: Only Brand Strategist called")
print("=" * 60)

single_session = agent_engine.create_session(user_id="workshop-test")

for event in agent_engine.stream_query(
    user_id="workshop-test",
    session_id=single_session["id"],
    message="Research the luxury skincare market on Instagram - who are the top brands and trends in 2025?",
):
    if "content" in event and "parts" in event["content"]:
        for part in event["content"]["parts"]:
            if "text" in part:
                print(part["text"], end="", flush=True)

print("\n" + "=" * 60)

---
## Step 18: Explore the Deployed System

Let's inspect the deployed agents and observe the A2A communication pattern.

In [ ]:
%%bash
# Check deployed Cloud Run services
gcloud run services list \
    --region=${REGION:-us-central1} \
    --format='table(metadata.name,status.url,status.conditions[0].status)'

In [ ]:
%%bash
# View agent cards for all deployed specialists
source .env

for agent_name in "STRATEGIST" "COPYWRITER" "DESIGNER" "CRITIC" "PM"; do
    url_var="${agent_name}_AGENT_URL"
    url=$(eval echo "\$$url_var")
    if [ -n "$url" ]; then
        echo "=== $agent_name Agent Card ==="
        curl -s "${url}/.well-known/agent.json" | python3 -c "
import sys, json
card = json.load(sys.stdin)
print(f\"  Name: {card.get('name')}\")
print(f\"  URL:  {card.get('url')}\")
skills = card.get('skills', [])
if skills:
    print(f\"  Skills: {', '.join(s.get('id','') for s in skills)}\")
"
        echo ""
    fi
done

In [ ]:
%%bash
# View Cloud Logging to observe A2A calls in action
# Run this WHILE or AFTER a campaign request is in progress

gcloud logging read \
    'resource.type="cloud_run_revision" AND textPayload:"brand_strategist"' \
    --limit=10 \
    --format='value(timestamp, textPayload)' \
    --freshness=1h

---
## Step 19: (Optional) Enable Notion Integration

The Project Manager can create tasks directly in **Notion** via MCP.

### Setup Steps

1. Go to [notion.so/my-integrations](https://www.notion.so/my-integrations) → **New Integration**
2. Name it `AI Creative Studio` → Copy the **Internal Integration Token**
3. Create TWO databases in Notion:
   - **Projects** (properties: Title, Status, Date range, Notes)
   - **Tasks** (properties: Title, Status, Due date, relation to Projects)
4. Share both databases with your integration (Share → Invite → select your integration)
5. Copy the **Projects database ID** from the URL: `notion.so/{workspace}/{DATABASE_ID}?v=...`

In [ ]:
# Configure Notion credentials
NOTION_API_KEY = "ntn_your-notion-token"        # <-- CHANGE THIS
NOTION_DATABASE_ID = "your-database-id"          # <-- CHANGE THIS

# Add to .env
with open(".env", "a") as f:
    f.write(f"\nNOTION_API_KEY={NOTION_API_KEY}")
    f.write(f"\nNOTION_DATABASE_ID={NOTION_DATABASE_ID}")

print("Notion credentials added to .env")

In [ ]:
%%bash
# Redeploy Project Manager with Notion credentials
source .env

echo "Redeploying Project Manager with Notion integration..."

gcloud run deploy project-manager-agent \
    --source agents/project_manager \
    --region ${GCP_REGION:-us-central1} \
    --allow-unauthenticated \
    --set-env-vars "GOOGLE_API_KEY=${GOOGLE_API_KEY},PROTOCOL=https,PUBLIC_PORT=443,NOTION_API_KEY=${NOTION_API_KEY},NOTION_DATABASE_ID=${NOTION_DATABASE_ID}"

echo "Project Manager redeployed with Notion!"

In [ ]:
# Re-run a campaign to see Notion tasks appear automatically
# After completion, check your Notion workspace for the project and tasks!

notion_session = agent_engine.create_session(user_id="workshop-notion-test")

print("Running campaign with Notion integration...")
print("Check your Notion workspace during/after this run!\n")
print("=" * 60)

for event in agent_engine.stream_query(
    user_id="workshop-notion-test",
    session_id=notion_session["id"],
    message="Create a complete campaign for GreenBrew eco-friendly coffee targeting Gen-Z",
):
    if "content" in event and "parts" in event["content"]:
        for part in event["content"]["parts"]:
            if "text" in part and "Notion" in part["text"]:
                print(part["text"], end="", flush=True)

print("\nDone! Check Notion for your new project and tasks.")

---
## Step 20: How the Critic Revision Loop Works

One of the most interesting patterns in this system is the **automatic quality improvement loop**. Let's trace how it works:

```
Creative Director
      │
      ├── calls Copywriter → receives 5 posts
      │
      ├── calls Designer → receives image concepts  
      │
      ├── calls Critic → receives structured review:
      │       POSTS: Score 6/10 - NEEDS_REVISION
      │       VISUALS: Score 8/10 - APPROVED
      │
      ├── [if NEEDS_REVISION] calls Copywriter AGAIN:
      │       "Revise posts. Original: [...] Critic feedback: [...]"
      │
      └── calls Project Manager with REVISED posts + APPROVED visuals
```

**Why maximum 1 revision?** To prevent runaway costs. After 1 revision, the orchestrator always proceeds regardless of the second critic score.

In [ ]:
# Explore the revision workflow in the orchestrator's system prompt
with open("agents/creative_director/agent.py") as f:
    content = f.read()

# Find and display the REVISION WORKFLOW section
lines = content.split("\n")
start = next((i for i, l in enumerate(lines) if "REVISION WORKFLOW" in l), None)
if start:
    end = next((i for i, l in enumerate(lines[start+1:], start+1) if l.strip().startswith('"""')), start+80)
    print("\n".join(lines[start:min(start+60, len(lines))]))

---
## Step 21: Cleanup Resources

When you're done with the workshop, clean up GCP resources to avoid ongoing charges.

In [ ]:
# Review what will be deleted before running cleanup
%%bash
source .env

echo "Resources to be deleted:"
echo ""
echo "Cloud Run services:"
gcloud run services list --region=${GCP_REGION:-us-central1} --format='value(metadata.name)'

echo ""
echo "Agent Engine:"
echo "  ID: $AGENT_ENGINE_ID"
echo "  Resource: $AGENT_ENGINE_RESOURCE_NAME"

In [ ]:
# Confirm before deleting
confirm = input("Type 'DELETE' to confirm cleanup, or press Enter to skip: ")

if confirm == "DELETE":
    print("Running teardown...")
    !bash deploy/teardown_gcp.sh
    print("Cleanup complete!")
else:
    print("Cleanup skipped. Resources are still running.")
    print("Remember to clean up manually to avoid charges!")

In [ ]:
%%bash
# Manual cleanup if teardown script is unavailable
source .env

REGION=${GCP_REGION:-us-central1}

# Delete Cloud Run services
for svc in brand-strategist-agent copywriter-agent designer-agent critic-agent project-manager-agent; do
    gcloud run services delete $svc --region=$REGION --quiet 2>/dev/null && echo "Deleted: $svc" || echo "Not found: $svc"
done

# Delete Agent Engine (orchestrator)
if [ -n "$AGENT_ENGINE_RESOURCE_NAME" ]; then
    python deploy/deploy_orchestrator.py --action cleanup
fi

echo "Manual cleanup complete!"

---
## Congratulations!

You've successfully built and deployed a **production-grade multi-agent AI system** on Google Cloud!

### What You Built

| Component | Technology | Location |
|---|---|---|
| Brand Strategist | ADK Agent + Google Search | Cloud Run |
| Copywriter | ADK Agent | Cloud Run |
| Designer | ADK Agent | Cloud Run |
| Critic | ADK Agent | Cloud Run |
| Project Manager | ADK Agent + Notion MCP | Cloud Run |
| Creative Director | ADK Orchestrator + A2A | Agent Engine |

### Key Patterns You Learned

1. **ADK Agent** — Define LLM agents with instructions + tools
2. **A2A Protocol** — Expose agents as discoverable HTTPS services  
3. **RemoteA2aAgent + AgentTool** — Orchestrate remote agents as callable tools
4. **MCP Integration** — Connect to external services (Notion) via stdio servers
5. **Context Compaction** — Handle long multi-agent workflows within token limits
6. **Quality Loop** — Structured critic feedback → conditional revision → proceed
7. **Cloud Run + Agent Engine** — Deploy agents at scale on GCP

### Next Steps

- Add **Imagen API** calls to the Designer to generate actual images
- Add **authentication** to Cloud Run services (remove `--allow-unauthenticated`)
- Swap one specialist for a **LangGraph or CrewAI** agent — A2A is framework-agnostic!
- Add a **feedback tool** in the orchestrator to let users rate outputs
- Explore **Vertex AI Agent Engine** tracing in the Cloud Console

### Resources

- [Google ADK Documentation](https://google.github.io/adk-docs/)
- [A2A Protocol Specification](https://google.github.io/A2A/)
- [Vertex AI Agent Engine](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview)
- [ADK Codelabs](https://codelabs.developers.google.com/?cat=aiml&text=adk)

---
*Workshop by AI Creative Studio — Built with Google ADK 1.20.0 and Gemini 2.5 Flash*